# Step 1: train and compare models (practice notebook)

Claude runs the real training from the `.py` files in `ml/`. This notebook lets you do the same steps by hand, in the same style as the step-0 notebook: read, run, then fill in the "Your turn" cells.

**The roadmap (what each section does)**

| Section | Question it answers | File behind it |
|---|---|---|
| 1. One patient's moments | What is one training row? | `labels.py`, `features.py` |
| 2. Labels | What are we trying to predict, and how rare is it? | `labels.py` |
| 3. Features | What may the model look at? | `features.py` |
| 4. Split by patient | How do we test honestly? | `split.py` |
| 5. Baselines | What is the bar to beat? | `baseline.py` |
| 6. LightGBM | How does it train, and when do we stop? | `model_lgbm.py` |
| 7. XGBoost vs LightGBM | Does the choice of booster matter? | `model_xgb.py` |

**Metrics, in one line each.** AUC: how often a real event gets a higher score than a safe moment (0.5 = coin flip). PR AUC: how clean the alerts are when events are rare (compare it with the event rate, not with 1.0).

In [ ]:
import sys
sys.path.append("..")            # so we can import the files in ml/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from baseline import prepare, TARGETS
from model_lgbm import cv_lgbm, per_fold_baselines
from model_xgb import cv_xgb

FOLDER = "../data/shanghai_t2dm_clean"
data, feats = prepare(FOLDER)    # builds labels + features + split (about a minute)
print("moments:", len(data), "  features:", len(feats))

## 1. One patient's moments

One row = one 15-minute look at a patient. The **features** are what we can see right now. The **labels** say what happened next (True = it happened, empty = cannot tell, for example readings are missing ahead).

In [ ]:
rid = "2075_0_20210720"
one = data[data["recording_id"] == rid].reset_index(drop=True)
cols = ["timestamp", "cgm_now", "cgm_change_60", "min_since_meal", "min_since_insulin", "low60", "spike120"]
one[cols].iloc[60:72]

In [ ]:
day = one.iloc[96:192]           # one day = 96 readings
plt.figure(figsize=(10, 3.5))
plt.plot(day["timestamp"], day["cgm_now"], color="black", lw=1.2)
hit = day[day["spike120"] == True]
plt.scatter(hit["timestamp"], hit["cgm_now"], color="darkorange", s=14, label="spike coming within 2 h")
plt.axhline(180, color="darkorange", ls="--", lw=1); plt.axhline(70, color="firebrick", ls="--", lw=1)
plt.legend(); plt.title(rid + ", one day"); plt.show()

**Your turn**
1. Change `rid` to `"2077_0_20210803"` and look at the table. What do you notice about `cgm_now`?
2. Find one row where `spike120` is True. Which features look different from a row where it is False?

In [ ]:
# Your turn: write your code here

## 2. Labels: how rare is each event?

In [ ]:
rows = []
for t in TARGETS:
    y = data[t].dropna().astype(bool)
    rows.append((t, len(y), int(y.sum()), round(100 * y.mean(), 2), data.loc[y[y].index, "patient_id"].nunique()))
pd.DataFrame(rows, columns=["label", "moments", "positives", "% positive", "patients with any"])

**Your turn**
1. Why is `low60` so rare (about 0.4%) compared with `spike120` (about 16%)? Give two reasons.
2. A model that never raises an alarm for lows would be about 99.6% "accurate". Why is that useless?

In [ ]:
# Your turn: write your code here

## 3. Features: what the model may look at

Rule: a feature may only use information from **now or the past**, never the future. `features.py` has a test that checks this (cut the data at a moment and the earlier features must not change).

In [ ]:
print(feats)
data[feats].describe().T[["count", "mean", "min", "max"]].round(1)

In [ ]:
data[feats].isna().mean().round(3).sort_values(ascending=False).head(6)

**Your turn**
1. Two features have missing values (`hba1c`, `egfr`). Why might they be missing, and how should a model treat them?
2. `min_since_meal` is capped at 1440 minutes. Why is "no meal in the last 24 hours" stored as a number instead of left empty?
3. Idea: add a feature `cgm_change_15` (sugar now minus 15 minutes ago). Why is that allowed but "sugar 15 minutes from now" is not?

In [ ]:
# Your turn: write your code here

## 4. Split by patient

Same person never in both train and test. 20 patients are locked away for a final test; the other 80 are in 5 folds.

In [ ]:
print(data.groupby("group")["patient_id"].nunique())
assert (data.groupby("patient_id")["group"].nunique() == 1).all()
print("OK: every patient is in exactly one group")

**Your turn:** What would go wrong if we split by rows (random moments) instead of by patients? Explain in one sentence.

In [ ]:
# Your turn: write your code here

## 5. Baselines: the bar to beat

The rule is "lower sugar = more worried" (for spikes: higher sugar). Logistic regression weighs all features. Numbers are mean AUC over the 5 folds.

In [ ]:
rows = []
for t in TARGETS:
    r, l = per_fold_baselines(data, feats, t)
    rows.append((t, round(r, 3), round(l, 3)))
pd.DataFrame(rows, columns=["label", "rule AUC", "logistic regression AUC"])

**Your turn:** For `low60` the plain rule beats logistic regression. Why might extra features not help when there are only about 300 events?

In [ ]:
# Your turn: write your code here

## 6. LightGBM: training and early stopping

We train on 3 folds, watch the error on a 4th fold (the practice exam), and stop when it stops improving. The 5th fold is only used for scoring.

In [ ]:
import lightgbm as lgb
from model_lgbm import PARAMS
t = "spike120"
d = data[(data["group"] != "test") & data[t].notna()]
tr, es = d[~d["group"].isin([0, 1])], d[d["group"] == 1]
hist = {}
m = lgb.train({**PARAMS, "metric": "binary_logloss"}, lgb.Dataset(tr[feats], tr[t].astype(int)), num_boost_round=1000,
              valid_sets=[lgb.Dataset(tr[feats], tr[t].astype(int)), lgb.Dataset(es[feats], es[t].astype(int))],
              valid_names=["practice", "unseen"],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.record_evaluation(hist)])
plt.figure(figsize=(7, 3.5))
plt.plot(hist["practice"]["binary_logloss"], label="practice moments")
plt.plot(hist["unseen"]["binary_logloss"], label="unseen patients")
plt.axvline(m.best_iteration, color="grey", ls=":"); plt.xlabel("trees added"); plt.ylabel("error"); plt.legend(); plt.show()
print("stopped at", m.best_iteration, "trees")

In [ ]:
imp = pd.Series(m.feature_importance("gain"), index=feats).sort_values(ascending=False)
(imp / imp.sum()).round(3).head(8)

**Your turn**
1. Change `num_leaves` (in `PARAMS`, try 5 and 60) and `learning_rate` (try 0.2). Does the best number of trees change? Does the unseen error get better or worse?
2. Which features matter most? Does that match what you would guess?

In [ ]:
# Your turn: write your code here

## 7. XGBoost vs LightGBM

Both build trees one after another, each fixing the last one's mistakes. XGBoost grows trees level by level (a bit more cautious); LightGBM grows the most useful branch first (faster, can overfit with little data). Mean AUC over folds from the full run:

| label | rule | logistic regression | LightGBM | XGBoost |
|---|---|---|---|---|
| low60 | 0.855 | 0.840 | 0.802 | 0.844 |
| spike120 | 0.694 | 0.780 | 0.802 | 0.797 |
| large_rise120 | 0.686 | 0.762 | 0.805 | 0.807 |

Re-run one label yourself:

In [ ]:
t = "spike120"
rate, auc, pr, rounds, per_fold = cv_lgbm(data, feats, t)
x_auc, x_pr, x_rounds = cv_xgb(data, feats, t)
print("LightGBM mean AUC", round(float(np.mean(per_fold)), 3), " XGBoost mean AUC", round(x_auc, 3))

**Your turn:** The two are almost the same for spikes. For lows XGBoost looks better but it is still below the simple rule. What would you try next for lows? (Ideas: fewer leaves, force "lower sugar = higher risk", more patients.)

In [ ]:
# Your turn: write your code here

## Done? Your step-1 checklist

- [ ] All cells run without errors
- [ ] You can say in your own words: AUC vs accuracy, and why early stopping is needed
- [ ] You can say why we split by patient
- [ ] You tried changing a LightGBM setting and saw what happened

**Send back to Claude:** your answers and anything that surprised you. Next steps: constrained model for lows, calibration, per-patient scores, then connect the model to the app.